# Python to SQL, and back again
In this codealong we will show you how to create a relational database from your pandas DataFrames.
> **To run this notebook you will need to work locally and not on colab.**

---
## 1.&nbsp; Import libraries 💾
If you haven't already installed sqlalchemy, you will need to. Uncomment the code below, install, and then recomment the code - you only need to install it once.

In [1]:
# install if needed
!conda install sqlalchemy -y
!conda install pymysql -y
!conda install sqlalchemy pymysql
!conda install python-dotenv -y
import os
from dotenv import load_dotenv

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 26.7.3
    latest version: 26.9.3

Please update conda by running

    $ conda self update



## Package Plan ##

  environment location: /home/sc4r3cr0w/anaconda3/envs/gans_scooter

  added / updated specs:
    - sqlalchemy


The following NEW packages will be INSTALLED:

  greenlet           pkgs/main/linux-64::greenlet-3.3.2-py314h7354ed3_0 
  sqlalchemy         pkgs/main/linux-64::sqlalchemy-2.0.52-py314h357f0b5_0 




Preparing transaction: done
Verifying transaction: done
Executing transaction: done
WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html

Jupyter detected...
2 channel Terms of Service ac

In [2]:
import pandas as pd

---
## 2.&nbsp; Relational Databases 📂

Creating DataFrames in python and pandas often results in tables with repeated information, as shown in the example below.
<br>

| author_name | book_title | year_published |
| --- | --- | --- |
| Arthur Conan Doyle | The Adventures of Sherlock Holmes | 1887 |
| J.R.R. Tolkien | The Hobbit | 1937 |
| J.R.R. Tolkien | The Lord of the Rings | 1954 |
| Harper Lee | To Kill a Mockingbird | 1960 |
| Harper Lee | Go Set a Watchman | 2015 |
<br>

This can be problematic for relational databases, which are designed to store data efficiently and avoid redundancy. To address this issue, we will separate the author and book information into two tables: authors and books. This approach eliminates duplicate data, ensuring data integrity and optimising storage.
<br>

| author_id | author_name |
| --- | --- |
| 1 | Arthur Conan Doyle |
| 2 | J.R.R. Tolkien |
| 3 | Harper Lee |
<br>

| book_id | book_title | year_published | author_id |
|---|---|---|---|
| 1 | The Adventures of Sherlock Holmes | 1887 | 1 |
| 2 | The Hobbit | 1937 | 2 |
| 3 | The Lord of the Rings | 1954 | 2 |
| 4 | To Kill a Mockingbird | 1960 | 3 |
| 5 | Go Set a Watchman | 2015 | 3 |

---
## 3.&nbsp; Creating the authors table with python 🐍
Let's start by creating the original DataFrame, including the repeated data.

In [3]:
names = ["Arthur Conan Doyle", "J.R.R. Tolkien", "J.R.R. Tolkien", "Harper Lee", "Harper Lee"]
titles = ["The Adventures of Sherlock Holmes", "The Hobbit", "The Lord of the Rings", "To Kill a Mockingbird", "Go Set a Watchman"]
years = [1887, 1937, 1954, 1960, 2015]

non_relational_df = pd.DataFrame({"author_name": names,
                                  "book_title": titles,
                                  "year_published": years})

non_relational_df

,author_name,book_title,year_published
0,Arthur Conan Doyle,The Adventures of Sherlock Holmes,1887
1,J.R.R. Tolkien,The Hobbit,1937
2,J.R.R. Tolkien,The Lord of the Rings,1954
3,Harper Lee,To Kill a Mockingbird,1960
4,Harper Lee,Go Set a Watchman,2015


Now, let's select only the authors without any duplicates.

In [4]:
authors_unique = non_relational_df["author_name"].unique()

authors_df = pd.DataFrame({"author_name": authors_unique})

authors_df

,author_name
0,Arthur Conan Doyle
1,J.R.R. Tolkien
2,Harper Lee


Fantastic! This DataFrame will be the foundation of our authors table.

---
## 4.&nbsp; Creating the matching authors table with SQL 💻

Ok, now we're ready to store this DataFrame in SQL. Before we can send the information in SQL, we need to make a table that has the same columns and data types to recieve the data. While we are creating a table for authors, we can also create the books table too.

Open MySQL Workbench, open a local connection, and open a new file. Then copy and paste the code from below.

```sql
-- Drop the database if it already exists
DROP DATABASE IF EXISTS sql_workshop ;

-- Create the database
CREATE DATABASE sql_workshop;

-- Use the database
USE sql_workshop;

-- Create the 'authors' table
CREATE TABLE authors (
    author_id INT AUTO_INCREMENT, -- Automatically generated ID for each author
    author_name VARCHAR(255) NOT NULL, -- Name of the author
    PRIMARY KEY (author_id) -- Primary key to uniquely identify each author
);

-- Create the 'books' table
CREATE TABLE books (
    book_id INT AUTO_INCREMENT, -- Automatically generated ID for each book
    book_title VARCHAR(255) NOT NULL, -- Title of the book
    year_published INT, -- Year the book was published
    author_id INT, -- ID of the author who wrote the book
    PRIMARY KEY (book_id), -- Primary key to uniquely identify each book
    FOREIGN KEY (author_id) REFERENCES authors(author_id) -- Foreign key to connect each book to its author
);
```

To download the sql file that we will follow for this section, [click here](https://drive.google.com/uc?export=download&id=1tln_33FM7D9wLckzxacBJNcMYqtyxybE)

If you'd like more information about MySQL data types [click here](https://www.w3schools.com/mysql/mysql_datatypes.asp).

---
## 5.&nbsp; Sending the information from this notebook to sql 📠
To establish a connection with the SQL database, we need to provide the notebook with the necessary information, which we do using the connection string below. You will need to modify only the password variable, which should match the password you set during MySQL Workbench installation.

In [5]:
# load.env-file
load_dotenv()

schema = "sql_authors"
host = "127.0.0.1"
user = "root"
password = os.getenv("DB_PASSWORD")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

To send information to our sql databse we use the pandas method `.to_sql()`. The argument `if_exists="append"` says that we don't want to overwrite any existing data, but add on to what is already there.

In [6]:
authors_df.to_sql('authors',
                  if_exists='append',
                  con=connection_string,
                  index=False)

3

Now, have a look at the table `authors` in MySQL Workbench, you should see that the names of the authors have appeared.

---
## 6.&nbsp; Retrieving information from sql to this notebook 📥
It's not only possible to send information to a SQL database, but also retrieve it too. Using `.read_sql()` in combination with the `connection_string` we can access the required data.

In [7]:
authors_from_sql = pd.read_sql("authors", con=connection_string)
authors_from_sql

,author_id,author_name
0,4,Arthur Conan Doyle
1,5,J.R.R. Tolkien
2,6,Harper Lee
3,7,Arthur Conan Doyle
4,8,J.R.R. Tolkien
5,9,Harper Lee
6,10,Arthur Conan Doyle
7,11,J.R.R. Tolkien
8,12,Harper Lee
9,13,Arthur Conan Doyle


Using this same method, we can also perform SQL queries to only bring back certain sections of information instead of the whole DataFrame.

In [8]:
pd.read_sql("""
            SELECT DISTINCT author_name
            FROM authors
            """,
            con=connection_string)

,author_name
0,Arthur Conan Doyle
1,J.R.R. Tolkien
2,Harper Lee


---
## 7.&nbsp; Preparing and sending the books table 📚
By extracting the authors table from our SQL database, we gain access to the unique identifier `author_id` assigned to each author. These `author_id`'s serve as pointers to their corresponding author records, allowing us to seamlessly link the `author_id`'s in the books table to their respective authors in the authors table, thereby completing the books table.

In [9]:
books_df = non_relational_df.merge(authors_from_sql,
                                   on = "author_name",
                                   how="left")

books_df

,author_name,book_title,year_published,author_id
0,Arthur Conan Doyle,The Adventures of Sherlock Holmes,1887,4
1,Arthur Conan Doyle,The Adventures of Sherlock Holmes,1887,7
2,Arthur Conan Doyle,The Adventures of Sherlock Holmes,1887,10
3,Arthur Conan Doyle,The Adventures of Sherlock Holmes,1887,13
4,Arthur Conan Doyle,The Adventures of Sherlock Holmes,1887,16
5,J.R.R. Tolkien,The Hobbit,1937,5
6,J.R.R. Tolkien,The Hobbit,1937,8
7,J.R.R. Tolkien,The Hobbit,1937,11
8,J.R.R. Tolkien,The Hobbit,1937,14
9,J.R.R. Tolkien,The Hobbit,1937,17


In [10]:
books_df = books_df.drop(columns=["author_name"])

books_df

,book_title,year_published,author_id
0,The Adventures of Sherlock Holmes,1887,4
1,The Adventures of Sherlock Holmes,1887,7
2,The Adventures of Sherlock Holmes,1887,10
3,The Adventures of Sherlock Holmes,1887,13
4,The Adventures of Sherlock Holmes,1887,16
5,The Hobbit,1937,5
6,The Hobbit,1937,8
7,The Hobbit,1937,11
8,The Hobbit,1937,14
9,The Hobbit,1937,17


In [11]:
books_df.to_sql('books',
                if_exists='append',
                con=connection_string,
                index=False)

25

In [12]:
books_from_sql = pd.read_sql("books", con=connection_string)
books_from_sql

,book_id,book_title,year_published,author_id
0,1,The Adventures of Sherlock Holmes,1887,4
1,2,The Adventures of Sherlock Holmes,1887,7
2,3,The Hobbit,1937,5
3,4,The Hobbit,1937,8
4,5,The Lord of the Rings,1954,5
5,6,The Lord of the Rings,1954,8
6,7,To Kill a Mockingbird,1960,6
7,8,To Kill a Mockingbird,1960,9
8,9,Go Set a Watchman,2015,6
9,10,Go Set a Watchman,2015,9


---
## 8.&nbsp; Challenge 😃
Now that you've learnt how to send and retrieve information, it's your turn to show off your skills. Create multiple tables in SQL for the data you scrapped about cities from Wikipedia. One should just be a table about the cities, the others should be facts about the cities.

| city_id | city |
| --- | --- |
| 1 | Berlin |
| 2 | Hamburg |
| 3 | Munich |

<br>

| City ID | Population | Year Data Retrieved |
|---|---|---|
| 1 | 3,850,809 | 2024 |
| 2 | 1,945,532 | 2024 |
| 3 | 1,512,491 | 2024 |

> **Pro Tip:** Visualise your relational database with pen and paper before you start coding. This can help you to identify any potential problems or inconsistencies in your design, and it can also make the coding process more efficient.

In [13]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [14]:
html_doc = """
<html><head><title>The Dormouse's story</title></head>
<body>
<p class="title"><b>The Dormouse's story</b></p>

<p class="story">Once upon a time there were three little sisters; and their names were
<a href="http://example.com/elsie" class="sister" id="link1" meta="Eldest sister">Elsie</a>,
<a href="http://example.com/lacie" class="sister" id="link2" meta="Middle sister">Lacie</a> and
<a href="http://example.com/tillie" class="sister" id="link3" meta="Youngest sister">Tillie</a>;
and they lived at the bottom of a well.</p>

<p class="story">...</p>
"""

In [15]:
soup = BeautifulSoup(html_doc, 'html.parser')

In [16]:
#Codes for navigating pursposes
last_link = soup.find(id='link3')
last_link

last_link.find_next()

second_to_last = last_link.find_previous()

In [26]:
city_to_scrape = "Berlin"
city_scraped = f"{city_to_scrape}"
url = f"https://en.wikipedia.org/wiki/{city_to_scrape}"
headers = {'User-Agent': 'Chrome/134.0.0.0'}

response = requests.get(url, headers=headers)

soup_3 = BeautifulSoup(response.content, 'html.parser')

city_soup = BeautifulSoup(response.content, "html.parser")

country = city_soup.find("th", string="Country").find_next().get_text()

# Latitude
lat = soup_3.find(class_="latitude").get_text(strip=True)

# Longitude
long = soup_3.find(class_="longitude").get_text(strip=True)


city_scraped, country, lat, long

('Berlin', 'Germany', '52°31′12″N', '13°24′18″E')

In [27]:
city = ["Berlin", "Hamburg", "Munich"]
country = ["city_soup.find("th", string="Country").find_next().get_text()", "The Hobbit", "The Lord of the Rings", "To Kill a Mockingbird", "Go Set a Watchman"]
lat_long = [1887, 1937, 1954, 1960, 2015]

cities_df = pd.DataFrame({"city": city,
                                  "country": country,
                                  "lat_long": lat_long})

cities_df

SyntaxError: invalid syntax. Is this intended to be part of the string? (871225302.py, line 2)

In [19]:
cities_unique = cities_df["city"].unique()

cities_df = pd.DataFrame({"city": cities_unique})

cities_df

,city
0,Arthur Conan Doyle
1,J.R.R. Tolkien
2,Harper Lee


In [20]:
# load.env-file
load_dotenv()

schema = "sql_cities"
host = "127.0.0.1"
user = "root"
password = os.getenv("DB_PASSWORD")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [21]:
cities_df.to_sql('cities',
                  if_exists='append',
                  con=connection_string,
                  index=False)

3